In [2]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [3]:
# ==========================================
# 1. GPU VRAM Safety Protocol
# ==========================================
# Forces TensorFlow to only use the VRAM it needs, preventing OOM crashes
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)

In [4]:
# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
CSV_FILE = r"C:\Study\IPD_Solar_Tracking\Dataset\CleanedDataSet.csv"                  # Your dataset CSV
IMAGE_DIR = r"C:\Study\IPD_Solar_Tracking\Dataset\Images"          # Folder containing the ESP32-CAM images
TARGET_COLUMN = "Global CMP22 (vent/cor) [W/m^2]"           # The CSV column with your W/m^2 values
IMAGE_FILENAME_COLUMN = "dateTime"     # The CSV column with your image names

IMG_SIZE = 380                         # Required for EfficientNetB4
BATCH_SIZE = 16                        # Keep low (8 or 16) if running on laptop CPU/GPU
EPOCHS = 30                            # Maximum training iterations
LEARNING_RATE = 0.001


In [9]:
# ==========================================
# 2. DATASET PREPARATION (Pandas) - UPDATED
# ==========================================
def prepare_dataframe():
    print("📊 Loading CSV dataset...")
    df = pd.read_csv(CSV_FILE)
    
    # 🚨 FIX 1: Force Pandas to treat the filename/dateTime column as strings
    df[IMAGE_FILENAME_COLUMN] = df[IMAGE_FILENAME_COLUMN].astype(str)
    
    # Get list of physical image files
    image_list = [filename[:12] for filename in os.listdir(IMAGE_DIR)]
    
    # Filter dataset to only include rows where the image actually exists
    df["ImageBool"] = df[IMAGE_FILENAME_COLUMN].isin(image_list)
    df = df[df["ImageBool"] == True]
    
    # Construct the full file paths
    df["file_path"] = df[IMAGE_FILENAME_COLUMN].apply(
        lambda x: os.path.join(IMAGE_DIR, f"{x}.jpg") # Adjust to .png if needed
    )
    
    # 🚨 FIX 2: Explicitly cast arrays to string and float32
    file_paths = df["file_path"].astype(str).values
    targets = df[TARGET_COLUMN].astype("float32").values
    
    print(f"✅ Found {len(df)} valid image-irradiance pairs.")
    return file_paths, targets

In [10]:



# ==========================================
# 3. TENSORFLOW DATA PIPELINE - UPDATED
# ==========================================
def process_path(file_path, irradiance):
    # Read the physical file
    img = tf.io.read_file(file_path)
    # Decode JPEG to a tensor
    img = tf.image.decode_jpeg(img, channels=3)
    # Resize to EfficientNetB4 specs (380x380)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    
    return img, irradiance

def create_tf_dataset(file_paths, targets):
    # 🚨 FIX 3: Cast dataset inputs to tf.string and tf.float32 explicitly
    dataset = tf.data.Dataset.from_tensor_slices((
        tf.cast(file_paths, tf.string),
        tf.cast(targets, tf.float32)
    ))
    
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

In [7]:
# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
def build_irradiance_model():
    print("⚙️ Building EfficientNetB4 Regression Architecture...")
    
    base_model = EfficientNetB4(
        weights='imagenet', 
        include_top=False, 
        input_shape=(IMG_SIZE, IMG_SIZE, 3) 
    )
    
    # Freeze the base model to train only the custom regression head first
    base_model.trainable = False 

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3), 
        
        # Single output neuron with ReLU (Irradiance cannot be negative)
        layers.Dense(1, activation='relu', name='irradiance_output') 
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.Huber(),
        metrics=['mae'] 
    )
    
    return model

In [12]:
# ==========================================
# 5. EXECUTION & TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    # 1. Prepare data
    paths, targets = prepare_dataframe()
    
    # Optional: Split into training and validation sets (80/20 split)
    split_index = int(len(paths) * 0.8)
    train_paths, val_paths = paths[:split_index], paths[split_index:]
    train_targets, val_targets = targets[:split_index], targets[split_index:]
    
    # 2. Build TensorFlow Datasets
    print("🚀 Building TensorFlow Data Pipelines...")
    train_dataset = create_tf_dataset(train_paths, train_targets)
    val_dataset = create_tf_dataset(val_paths, val_targets)
    
    # 3. Build Model
    model = build_irradiance_model()
    model.summary()
    
    # 4. Set up Callbacks
    # Saves the best version of the model automatically
    checkpoint = ModelCheckpoint(
        "efficientnet_b4_solar.keras", 
        monitor="val_loss", 
        save_best_only=True, 
        verbose=1
    )
    # Stops training early if the model stops improving, saving time
    early_stop = EarlyStopping(
        monitor="val_loss", 
        patience=5, 
        restore_best_weights=True
    )
    
    # 5. Train the Model
    print("🔥 Starting Training...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stop]
    )
    
    print("🎉 Training Complete! Model saved as 'efficientnet_b4_solar.keras'")

📊 Loading CSV dataset...
✅ Found 205 valid image-irradiance pairs.
🚀 Building TensorFlow Data Pipelines...
⚙️ Building EfficientNetB4 Regression Architecture...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb4 (Functional)     │ (None, 12, 12, 1792)   │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       459,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ irradiance_output (Dense)       │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,133,088 (69.17 MB)

 Trainable params: 459,265 (1.75 MB)

 Non-trainable params: 17,673,823 (67.42 MB)

🔥 Starting Training...
Epoch 1/30


c:\Study\IPD_Solar_Tracking\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 9/11 ━━━━━━━━━━━━━━━━━━━━ 7s 4s/step - loss: 237.5667 - mae: 238.0665 

KeyboardInterrupt: 